# 🚗 UNLET-ADAS: Low-Light Video Enhancement
### B.E. Major Project | SJBIT Bengaluru | CSE 2025-26
**GitHub:** https://github.com/DEEK-SHITH/UNLET-ADAS
---
**Run cells top to bottom. Do not skip any cell.**

In [ ]:
# ============================================================
# CELL 1 — Setup
# ============================================================
# Anti-disconnect
from IPython.display import display, Javascript
display(Javascript('''
function ClickConnect(){
    var btns = document.querySelectorAll("colab-toolbar-button");
    for(var i=0;i<btns.length;i++){
        if(btns[i].id=="connect") btns[i].click();
    }
}
setInterval(ClickConnect, 55000)
'''))
print('Anti-disconnect active!')

# Mount Drive
from google.colab import drive
drive.mount('/content/drive')

# Install packages
!pip install ultralytics deep-sort-realtime scikit-image timm -q

# Clone repo
import os
if not os.path.exists('/content/UNLET-ADAS'):
    !git clone https://github.com/DEEK-SHITH/UNLET-ADAS.git /content/UNLET-ADAS
else:
    !cd /content/UNLET-ADAS && git pull

import sys
sys.path.insert(0, '/content/UNLET-ADAS')

import numpy as np
import cv2
import torch
import torch.nn.functional as F
import torchvision.models as tvm
from PIL import Image, ImageOps
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from glob import glob
import time, json, warnings
warnings.filterwarnings('ignore')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
print('Setup complete!')

In [ ]:
# ============================================================
# CELL 2 — Configuration
# ============================================================
SAVE_DIR    = '/content/drive/MyDrive/UNLET_Project/checkpoints'
OUTPUT_DIR  = '/content/drive/MyDrive/UNLET_Project/results'
VIDEO_INPUT = '/content/drive/MyDrive/UNLET_Project/night_drive.mp4'
LOL_ROOT    = '/content/drive/MyDrive/UNLET_Project/lol_dataset'
WEIGHTS     = os.path.join(SAVE_DIR, 'zerodce_cbam_best.pt')
IMAGE_SIZE  = 256

os.makedirs(SAVE_DIR,   exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

print('Checking paths...')
paths = {
    'Video'      : VIDEO_INPUT,
    'LOL train'  : os.path.join(LOL_ROOT,'our485','low'),
    'LOL val'    : os.path.join(LOL_ROOT,'eval15','low'),
    'Save dir'   : SAVE_DIR,
}
for label, path in paths.items():
    exists = os.path.exists(path)
    info   = (f'{len(os.listdir(path))} files'
              if exists and os.path.isdir(path) else
              'found' if exists else 'MISSING')
    print(f'  {"OK" if exists else "MISSING"} | {label}: {info}')
print('Config done!')

In [ ]:
# ============================================================
# CELL 3 — Build Model
# ============================================================
from src.model import build_model

model = build_model().to(DEVICE)

# Verify
with torch.no_grad():
    t = torch.rand(2,3,256,256).to(DEVICE)
    e, c = model(t)
print(f'Input  : {t.shape}')
print(f'Output : {e.shape}')
print(f'Curves : {c.shape}')
print('Model ready!')

In [ ]:
# ============================================================
# CELL 4 — Loss Functions
# ============================================================
from src.losses import UNLETLoss

criterion = UNLETLoss(device=str(DEVICE)).to(DEVICE)

# Sanity check
with torch.no_grad():
    d = torch.rand(2,3,256,256).to(DEVICE)
    e, c = model(d)
    l = criterion(e, c, d, d)
print(f'Loss check: {float(l):.4f}')

# Verify color loss is strong
print(f'Color constancy weight : 50.0 (prevents green tint)')
print(f'SSIM weight            : 2.0')
print(f'Perceptual weight      : 0.1')
print('Loss functions ready!')

In [ ]:
# ============================================================
# CELL 5 — Load Dataset and Train
# ============================================================
from src.train import LOLDataset, compute_metrics
from torch.utils.data import DataLoader
from torch.optim.lr_scheduler import CosineAnnealingLR

# Copy dataset to local for speed
import shutil
LOCAL_LOL = '/content/lol_dataset'
if not os.path.exists(f'{LOCAL_LOL}/our485/low') or \
   len(os.listdir(f'{LOCAL_LOL}/our485/low'))==0:
    print('Copying dataset to local storage...')
    for split in ['our485/low','our485/high','eval15/low','eval15/high']:
        src = os.path.join(LOL_ROOT, split)
        dst = os.path.join(LOCAL_LOL, split)
        os.makedirs(dst, exist_ok=True)
        shutil.copytree(src, dst, dirs_exist_ok=True)
        print(f'  {split}: {len(os.listdir(dst))} images')
else:
    print('Dataset already in local storage')

print('\nLoading datasets...')
train_ds = LOLDataset(f'{LOCAL_LOL}/our485/low',
                      f'{LOCAL_LOL}/our485/high')
val_ds   = LOLDataset(f'{LOCAL_LOL}/eval15/low',
                      f'{LOCAL_LOL}/eval15/high')
train_dl = DataLoader(train_ds, batch_size=8, shuffle=True,
                      num_workers=2, pin_memory=True)
val_dl   = DataLoader(val_ds,   batch_size=4, shuffle=False,
                      num_workers=2, pin_memory=True)
print(f'Train: {len(train_ds)} images | Val: {len(val_ds)} images')

# Training config
EPOCHS   = 100
PATIENCE = 20
opt      = torch.optim.Adam(model.parameters(), lr=2e-4, weight_decay=1e-5)
sched    = CosineAnnealingLR(opt, T_max=EPOCHS, eta_min=1e-6)
best_val = float('inf')
patience = 0
history  = {'train':[],'val':[],'psnr':[],'ssim':[]}

print(f'\nStarting training: {EPOCHS} epochs')
print('-'*65)
t0 = time.time()

for epoch in range(EPOCHS):
    model.train()
    tl = []
    for low, high in train_dl:
        low, high = low.to(DEVICE), high.to(DEVICE)
        opt.zero_grad()
        enh, curves = model(low)
        norm = high if high.sum()>0 else None
        loss = criterion(enh, curves, low, norm)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        tl.append(loss.item())
    sched.step()

    model.eval()
    vl, vp, vs = [], [], []
    with torch.no_grad():
        for low, high in val_dl:
            low, high = low.to(DEVICE), high.to(DEVICE)
            enh, curves = model(low)
            norm = high if high.sum()>0 else None
            vl.append(criterion(enh, curves, low, norm).item())
            if high.sum()>0:
                p, s = compute_metrics(enh, high)
                vp.append(p); vs.append(s)

    tl_m=float(np.mean(tl)); vl_m=float(np.mean(vl))
    vp_m=float(np.mean(vp)) if vp else 0.0
    vs_m=float(np.mean(vs)) if vs else 0.0
    elapsed=(time.time()-t0)/60

    history['train'].append(tl_m); history['val'].append(vl_m)
    history['psnr'].append(vp_m);  history['ssim'].append(vs_m)

    if vl_m < best_val:
        best_val=vl_m; patience=0
        torch.save(model.state_dict(), WEIGHTS)
        marker=f'  SAVED  PSNR={vp_m:.2f}dB SSIM={vs_m:.4f}'
    else:
        patience+=1
        marker=f'  patience {patience}/{PATIENCE}'

    print(f'Ep {epoch+1:3d}/{EPOCHS} | '
          f'loss={tl_m:.3f} | val={vl_m:.3f} | '
          f'{elapsed:.1f}m{marker}')

    if patience>=PATIENCE:
        print(f'Early stopping at epoch {epoch+1}'); break

with open(os.path.join(SAVE_DIR,'history.json'),'w') as f:
    json.dump(history, f, indent=2)
print(f'\nDone! Best PSNR={max(history["psnr"]):.2f}dB SSIM={max(history["ssim"]):.4f}')

In [ ]:
# ============================================================
# CELL 6 — Load Best Weights + Quick Color Check
# ============================================================
model.load_state_dict(torch.load(WEIGHTS, map_location=DEVICE))
model.eval()
print('Best weights loaded!')

# Color check — verify no green tint
test = np.zeros((256,256,3), dtype=np.uint8)+20
t    = torch.from_numpy(test.astype(np.float32)/255.0
       ).permute(2,0,1).unsqueeze(0).to(DEVICE)
with torch.no_grad():
    enh, _ = model(t)
enh_np = (enh[0].permute(1,2,0).cpu().numpy()*255).clip(0,255).astype(np.uint8)

r = enh_np[:,:,0].mean()
g = enh_np[:,:,1].mean()
b = enh_np[:,:,2].mean()
diff = max(abs(r-g), abs(r-b), abs(g-b))

print(f'Color check: R={r:.1f} G={g:.1f} B={b:.1f}')
print(f'Max channel difference: {diff:.1f}')
if diff < 15:
    print('Colors are NEUTRAL — no green tint!')
elif diff < 30:
    print('Slight color cast — acceptable for demo')
else:
    print('Color cast detected — consider retraining with higher color_loss')

In [ ]:
# ============================================================
# CELL 7 — Test on LOL Images (Original vs AutoContrast vs UNLET)
# ============================================================
from src.enhance import enhance_image

def compare_plot(img_path, save_path=None):
    orig  = Image.open(img_path).convert('RGB').resize((256,256))
    auto  = ImageOps.autocontrast(orig)

    # Enhance
    arr   = np.array(orig,dtype=np.float32)/255.0
    t     = torch.from_numpy(arr).permute(2,0,1).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        enh,_ = model(t)
    enh_np = (enh[0].permute(1,2,0).cpu().numpy()*255).clip(0,255).astype(np.uint8)
    enh_pil = Image.fromarray(enh_np)

    # Brightness values
    ob = np.array(orig).mean()/255
    ab = np.array(auto).mean()/255
    eb = enh_np.mean()/255

    fig, axs = plt.subplots(1,3,figsize=(15,5))
    fig.patch.set_facecolor('#0a0f1e')
    for ax,(img,title,bright,color) in zip(axs,[
        (orig,    f'Original\nBrightness: {ob:.3f}',   ob, '#94a3b8'),
        (auto,    f'AutoContrast\nBrightness: {ab:.3f}',ab, '#f59e0b'),
        (enh_pil, f'UNLET Enhanced\nBrightness: {eb:.3f}',eb,'#22c55e')]):
        ax.imshow(img)
        ax.set_title(title, color=color,
                     fontsize=12, fontweight='bold', pad=8)
        ax.axis('off')
        ax.set_facecolor('#0a0f1e')
    fname = os.path.basename(img_path)
    fig.suptitle(f'UNLET-ADAS Enhancement — {fname}',
                 color='white', fontsize=13, y=1.02)
    plt.tight_layout(pad=0.5)
    if save_path:
        plt.savefig(save_path,dpi=150,bbox_inches='tight',facecolor='#0a0f1e')
    plt.show(); plt.close()

test_imgs = sorted(glob(f'{LOCAL_LOL}/eval15/low/*.png'))[:5]
img_results_dir = os.path.join(OUTPUT_DIR,'image_results')
os.makedirs(img_results_dir, exist_ok=True)

print(f'Testing on {len(test_imgs)} validation images...')
for i, p in enumerate(test_imgs):
    compare_plot(p, save_path=os.path.join(img_results_dir,f'result_{i+1:02d}.png'))
print(f'Results saved to: {img_results_dir}')

In [ ]:
# ============================================================
# CELL 8 — Evaluate PSNR and SSIM
# ============================================================
from skimage.metrics import peak_signal_noise_ratio as calc_psnr
from skimage.metrics import structural_similarity  as calc_ssim

def evaluate(model, low_dir, high_dir):
    lows  = sorted(glob(os.path.join(low_dir,'*.png')))
    highs = sorted(glob(os.path.join(high_dir,'*.png')))
    hmap  = {os.path.basename(p):p for p in highs}

    psnr_orig,ssim_orig = [],[]
    psnr_auto,ssim_auto = [],[]
    psnr_enh, ssim_enh  = [],[]

    for lp in lows:
        name = os.path.basename(lp)
        if name not in hmap: continue

        low_np  = np.array(Image.open(lp).convert('RGB').resize((256,256)),dtype=np.float32)/255.0
        high_np = np.array(Image.open(hmap[name]).convert('RGB').resize((256,256)),dtype=np.float32)/255.0
        auto_np = np.array(ImageOps.autocontrast(
            Image.open(lp).convert('RGB').resize((256,256))),dtype=np.float32)/255.0

        t   = torch.from_numpy(low_np).permute(2,0,1).unsqueeze(0).to(DEVICE)
        with torch.no_grad(): enh,_ = model(t)
        enh_np = enh[0].permute(1,2,0).cpu().numpy().clip(0,1)

        psnr_orig.append(calc_psnr(high_np,low_np, data_range=1.0))
        psnr_auto.append(calc_psnr(high_np,auto_np,data_range=1.0))
        psnr_enh.append( calc_psnr(high_np,enh_np, data_range=1.0))
        ssim_orig.append(calc_ssim(high_np,low_np, channel_axis=2,data_range=1.0))
        ssim_auto.append(calc_ssim(high_np,auto_np,channel_axis=2,data_range=1.0))
        ssim_enh.append( calc_ssim(high_np,enh_np, channel_axis=2,data_range=1.0))

    results = {
        'Original'    :{'psnr':float(np.mean(psnr_orig)),'ssim':float(np.mean(ssim_orig))},
        'AutoContrast':{'psnr':float(np.mean(psnr_auto)),'ssim':float(np.mean(ssim_auto))},
        'UNLET (Ours)':{'psnr':float(np.mean(psnr_enh)), 'ssim':float(np.mean(ssim_enh))},
    }
    return results

print('Evaluating on LOL eval15...')
results = evaluate(model,
                   f'{LOCAL_LOL}/eval15/low',
                   f'{LOCAL_LOL}/eval15/high')

print('\n'+'='*48)
print('  EVALUATION RESULTS — LOL eval15')
print('='*48)
print(f'  {"Method":<16}  {"PSNR(dB)":>9}  {"SSIM":>8}')
print('-'*48)
for method,scores in results.items():
    m = ' <- Our Model' if 'UNLET' in method else ''
    print(f'  {method:<16}  {scores["psnr"]:>9.2f}  {scores["ssim"]:>8.4f}{m}')
print('='*48)

with open(os.path.join(OUTPUT_DIR,'eval_results.json'),'w') as f:
    json.dump(results,f,indent=2)

# Bar chart
methods = list(results.keys())
psnrs   = [results[m]['psnr'] for m in methods]
ssims   = [results[m]['ssim'] for m in methods]
colors  = ['#94a3b8','#f59e0b','#22c55e']

fig,axes = plt.subplots(1,2,figsize=(13,5))
fig.patch.set_facecolor('#0f172a')
for ax,vals,ylabel,title in zip(axes,
    [psnrs,ssims],['PSNR (dB)','SSIM'],
    ['PSNR Comparison','SSIM Comparison']):
    ax.set_facecolor('#1e293b')
    bars=ax.bar(methods,vals,color=colors,width=0.5,edgecolor='white')
    for bar,v in zip(bars,vals):
        ax.text(bar.get_x()+bar.get_width()/2,
                bar.get_height()+max(vals)*0.01,
                f'{v:.3f}',ha='center',color='white',
                fontsize=12,fontweight='bold')
    ax.set_title(title,color='white',fontsize=12,fontweight='bold')
    ax.set_ylabel(ylabel,color='white')
    ax.tick_params(colors='white')
    ax.spines[['top','right','left','bottom']].set_color('#334155')
    ax.set_ylim(0,max(vals)*1.18)
fig.suptitle('UNLET-ADAS vs Baselines',color='white',fontsize=14,fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR,'metric_comparison.png'),
            dpi=150,bbox_inches='tight',facecolor='#0f172a')
plt.show()
print('Chart saved!')

In [ ]:
# ============================================================
# CELL 9 — Enhance Night Driving Video
# Outputs:
#   1. enhanced_night_drive.mp4  (enhanced only)
#   2. original_night_drive.mp4  (original only)
#   3. comparison_night_drive.mp4 (side by side)
# ============================================================
from src.enhance import enhance_video

ENH_OUTPUT  = os.path.join(OUTPUT_DIR,'enhanced_night_drive.mp4')
ORIG_OUTPUT = os.path.join(OUTPUT_DIR,'original_night_drive.mp4')
CMP_OUTPUT  = os.path.join(OUTPUT_DIR,'comparison_night_drive.mp4')

if not os.path.exists(VIDEO_INPUT):
    print(f'Video not found: {VIDEO_INPUT}')
    print('Upload night_drive.mp4 to your Drive first')
else:
    print('Starting video enhancement...')
    print('This will produce 3 separate video files')
    print()

    total_frames = enhance_video(
        model         = model,
        device        = DEVICE,
        input_path    = VIDEO_INPUT,
        output_path   = ENH_OUTPUT,
        original_path = ORIG_OUTPUT,
        comparison_path = CMP_OUTPUT,
        batch_size    = 8,
        size          = 256
    )

    print(f'\n3 video files saved to Drive:')
    for f in [ENH_OUTPUT, ORIG_OUTPUT, CMP_OUTPUT]:
        size = os.path.getsize(f)/(1024*1024)
        print(f'  {os.path.basename(f):<40} {size:.1f} MB')

In [ ]:
# ============================================================
# CELL 10 — Show Before/After Video Frames
# ============================================================
def show_video_comparison(video_path, n=5, save_path=None):
    if not os.path.exists(video_path):
        print(f'Video not found: {video_path}'); return

    cap   = cv2.VideoCapture(video_path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    Wh    = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))//2
    idxs  = np.linspace(int(total*0.05),int(total*0.95),n,dtype=int)

    fig = plt.figure(figsize=(n*4,9))
    fig.patch.set_facecolor('#0a0f1e')
    fig.suptitle('UNLET-ADAS: Night Driving Enhancement\n'
                 'Original (top) vs UNLET Enhanced (bottom)',
                 color='white',fontsize=13,fontweight='bold',y=1.02)
    gs = gridspec.GridSpec(2,n,figure=fig,hspace=0.06,wspace=0.04)

    for col,idx in enumerate(idxs):
        cap.set(cv2.CAP_PROP_POS_FRAMES,int(idx))
        ret,frame = cap.read()
        if not ret: continue
        orig = cv2.cvtColor(frame[:,:Wh],cv2.COLOR_BGR2RGB)
        enh  = cv2.cvtColor(frame[:,Wh:],cv2.COLOR_BGR2RGB)

        ax0 = fig.add_subplot(gs[0,col])
        ax0.imshow(orig); ax0.axis('off')
        ax0.set_title(f'Frame {idx}',color='#64748b',fontsize=8)
        if col==0:
            ax0.set_ylabel('Original',color='#ef4444',
                           fontsize=11,fontweight='bold',rotation=90)

        ax1 = fig.add_subplot(gs[1,col])
        ax1.imshow(enh); ax1.axis('off')
        if col==0:
            ax1.set_ylabel('UNLET Enhanced',color='#22c55e',
                           fontsize=11,fontweight='bold',rotation=90)

    cap.release()
    plt.tight_layout(pad=0.2)
    if save_path:
        plt.savefig(save_path,dpi=150,bbox_inches='tight',facecolor='#0a0f1e')
        print(f'Saved: {save_path}')
    plt.show()

show_video_comparison(
    CMP_OUTPUT,
    save_path=os.path.join(OUTPUT_DIR,'video_frames.png'))

In [ ]:
# ============================================================
# CELL 11 — Training Curves + Final Summary
# ============================================================
fig,axes = plt.subplots(1,2,figsize=(14,5))
fig.patch.set_facecolor('#0f172a')
ep = range(1,len(history['train'])+1)

for ax in axes:
    ax.set_facecolor('#1e293b'); ax.tick_params(colors='white')
    ax.spines[['top','right','left','bottom']].set_color('#334155')

axes[0].plot(ep,history['train'],color='#38bdf8',lw=2,label='Train')
axes[0].plot(ep,history['val'],  color='#f59e0b',lw=2,ls='--',label='Val')
axes[0].set_title('Training Loss',color='white',fontsize=12,fontweight='bold')
axes[0].set_xlabel('Epoch',color='white'); axes[0].set_ylabel('Loss',color='white')
axes[0].legend(facecolor='#1e293b',labelcolor='white')

axes[1].plot(ep,history['psnr'],color='#22c55e',lw=2,label='PSNR (dB)')
ax2 = axes[1].twinx()
ax2.plot(ep,history['ssim'],color='#f59e0b',lw=2,ls='--',label='SSIM')
ax2.tick_params(colors='white'); ax2.set_ylabel('SSIM',color='white')
axes[1].set_title('PSNR & SSIM',color='white',fontsize=12,fontweight='bold')
axes[1].set_xlabel('Epoch',color='white'); axes[1].set_ylabel('PSNR (dB)',color='white')
axes[1].legend(loc='lower right',facecolor='#1e293b',labelcolor='white')
ax2.legend(loc='center right',facecolor='#1e293b',labelcolor='white')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR,'training_curves.png'),
            dpi=150,bbox_inches='tight',facecolor='#0f172a')
plt.show()

print('\n'+'='*55)
print('  UNLET-ADAS FINAL SUMMARY')
print('='*55)
print(f'  Model      : Zero-DCE++ + CBAM Attention')
print(f'  Dataset    : LOL (485 train / 15 test)')
print(f'  Epochs     : {len(history["train"])}')
print(f'  Best PSNR  : {max(history["psnr"]):.2f} dB')
print(f'  Best SSIM  : {max(history["ssim"]):.4f}')
print(f'  GitHub     : github.com/DEEK-SHITH/UNLET-ADAS')
print('='*55)
print('\nOutput files saved to Drive:')
for f in sorted(os.listdir(OUTPUT_DIR)):
    sz = os.path.getsize(os.path.join(OUTPUT_DIR,f))/(1024*1024)
    print(f'  {f:<45} {sz:6.1f} MB')
print('='*55)